# Analise Descritiva de Precos de Cafe

Notebook limpo para executar o projeto de ponta a ponta: leitura das bases, tratamento, normalizacao de marcas, normalizacao por peso, geracao dos CSVs finais e graficos.


## 1. Imports e caminhos


In [ ]:
from pathlib import Path
import re
import runpy

import numpy as np
import pandas as pd

BASE_DIR = Path("base")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


## 2. Leitura das bases


In [ ]:
mambo = pd.read_csv(BASE_DIR / "mambo.csv")
marche = pd.read_csv(BASE_DIR / "marche.csv")
base_cafe = pd.read_csv(BASE_DIR / "base_cafe.csv")

print("Mambo:", mambo.shape)
print("St Marche:", marche.shape)
print("Base Cafe/Pao de Acucar:", base_cafe.shape)


## 3. Funcoes de tratamento


In [ ]:
MAPA_MARCAS = {
    "L'OR": "L'OR",
    "L'Or": "L'OR",
    "Lor": "L'OR",
    "ORFEU": "Orfeu",
    "Orfeu": "Orfeu",
    "Santa Monica": "Santa Mônica",
    "Santa Mônica": "Santa Mônica",
    "Bravo Cafe": "Bravo Café",
    "Bravo Café": "Bravo Café",
    "Tres Coracoes": "3 Corações",
    "Três": "3 Corações",
    "Tres": "3 Corações",
    "3 Corações": "3 Corações",
    "QUALITA": "Qualitá",
    "Qualitá": "Qualitá",
    "Qualitá Exclusive": "Qualitá",
    "Prima Qualita": "Prima Qualitá",
    "Prima Qualitá": "Prima Qualitá",
    "Café Do Ponto": "Café do Ponto",
    "Café do Ponto": "Café do Ponto",
    "do Ponto": "Café do Ponto",
    "Cafe Iguacu": "Iguaçu",
    "Iguaçu": "Iguaçu",
    "pa": None,
}


def to_float(valor):
    return float(str(valor).replace(",", "."))


def extrair_quantidade(titulo):
    texto = str(titulo).lower()
    texto = texto.replace("º", " graus").replace("°", " graus")

    unidades = np.nan
    padrao_unidades = r"(\d+)\s*(?:unidades|unidade|unid\.?|capsulas|cápsulas|capsula|cápsula)\b"
    achou_unidade = re.search(padrao_unidades, texto)
    if achou_unidade:
        unidades = float(achou_unidade.group(1))

    padrao_cada = (
        r"(\d+)\s*(?:unidades|unidade|unid\.?|capsulas|cápsulas|capsula|cápsula)\b"
        r".*?(\d+(?:[\.,]\d+)?)\s*g\s*cada"
    )
    achou_cada = re.search(padrao_cada, texto)
    if achou_cada:
        unidades_cada = float(achou_cada.group(1))
        peso_unitario = to_float(achou_cada.group(2))
        return unidades_cada * peso_unitario, "g", unidades_cada

    achou_kg = re.search(r"(\d+(?:[\.,]\d+)?)\s*kg\b", texto)
    if achou_kg:
        return to_float(achou_kg.group(1)) * 1000, "kg", unidades

    achou_g = re.search(r"(\d+(?:[\.,]\d+)?)\s*g\b", texto)
    if achou_g:
        return to_float(achou_g.group(1)), "g", unidades

    return np.nan, np.nan, unidades


def faixa_peso(peso_g):
    if pd.isna(peso_g):
        return "sem_peso"
    if peso_g <= 100:
        return "ate_100g"
    if peso_g <= 250:
        return "101g_250g"
    if peso_g <= 500:
        return "251g_500g"
    if peso_g <= 1000:
        return "501g_1kg"
    return "acima_1kg"


## 4. Consolidacao das bases


In [ ]:
mambo_base = mambo.rename(
    columns={"SKU": "sku", "Titulo": "titulo", "Fabricante": "fabricante", "Preco": "preco"}
).copy()
mambo_base["loja"] = "Mambo"
mambo_base = mambo_base.rename(
    columns={"Categoria": "categoria", "Disponibilidade": "disponibilidade", "URL": "url"}
)
mambo_base = mambo_base[["loja", "sku", "titulo", "fabricante", "categoria", "disponibilidade", "preco", "url"]]

marche_base = marche.rename(
    columns={"SKU": "sku", "Titulo": "titulo", "Fabricante": "fabricante", "Preco": "preco"}
).copy()
marche_base["loja"] = "St Marche"
marche_base = marche_base.rename(
    columns={"Categoria": "categoria", "Disponibilidade": "disponibilidade", "URL": "url"}
)
marche_base = marche_base[["loja", "sku", "titulo", "fabricante", "categoria", "disponibilidade", "preco", "url"]]

pao_base = (
    base_cafe
    .sort_values("Data")
    .groupby("Código")
    .last()
    .reset_index()
)
pao_base = pao_base.rename(
    columns={
        "Código": "sku",
        "Descrição": "titulo",
        "Marca": "fabricante",
        "Categoria/Setor": "categoria",
        "Média Preço Normal": "preco",
        "URL Produto Monitorado": "url",
    }
)
pao_base["loja"] = "Paodeacucar"
pao_base["disponibilidade"] = np.where(
    pao_base["Leituras com disponibilidade"] > 0,
    "Disponivel",
    "Indisponivel",
)
pao_base = pao_base[["loja", "sku", "titulo", "fabricante", "categoria", "disponibilidade", "preco", "url"]]

base_total = pd.concat([mambo_base, marche_base, pao_base], ignore_index=True)
base_total["preco"] = pd.to_numeric(base_total["preco"], errors="coerce")

base_total.head()


## 5. Analises descritivas principais


In [ ]:
base_preco = base_total.dropna(subset=["preco"]).copy()

analise_loja = (
    base_preco
    .groupby("loja")["preco"]
    .agg(["mean", "median", "min", "max", "count"])
    .round(2)
    .sort_values("mean", ascending=False)
    .reset_index()
    .rename(columns={"loja": "Loja"})
)
analise_loja.to_csv(RESULTS_DIR / "analise_preco_loja.csv", index=False)
analise_loja


In [ ]:
analise_distribuicao = base_preco["preco"].describe().round(2).reset_index()
analise_distribuicao.columns = ["Describe", "Preco"]
analise_distribuicao.to_csv(RESULTS_DIR / "analise_distribuicao_preco.csv", index=False)
analise_distribuicao


## 6. Normalizacao de fabricantes


In [ ]:
base_marcas = base_total.copy()
base_marcas["fabricante_original"] = base_marcas["fabricante"]
base_marcas["fabricante"] = base_marcas["fabricante"].replace(MAPA_MARCAS)
base_marcas = base_marcas[base_marcas["fabricante"].notna()].copy()

analise_mix = pd.crosstab(base_marcas["fabricante"], base_marcas["loja"]).reset_index()
for coluna in ["Mambo", "St Marche", "Paodeacucar"]:
    if coluna not in analise_mix.columns:
        analise_mix[coluna] = 0
analise_mix = analise_mix.rename(columns={"fabricante": "Fabricante", "St Marche": "St.Marche"})
analise_mix["Total"] = analise_mix[["Mambo", "St.Marche", "Paodeacucar"]].sum(axis=1)
analise_mix = analise_mix[["Fabricante", "Mambo", "St.Marche", "Paodeacucar", "Total"]]
analise_mix = analise_mix.sort_values("Total", ascending=False)
analise_mix.to_csv(RESULTS_DIR / "analise_mix.csv", index=False)
analise_mix.head(10)


In [ ]:
analise_fabricante = (
    base_marcas
    .dropna(subset=["preco"])
    .groupby("fabricante")["preco"]
    .agg(["mean", "median", "min", "max", "count"])
    .sort_values("mean")
    .reset_index()
    .rename(columns={"fabricante": "Fabricante"})
)
analise_fabricante["range"] = analise_fabricante["max"] - analise_fabricante["min"]
colunas_preco = ["mean", "median", "min", "max", "range"]
analise_fabricante[colunas_preco] = analise_fabricante[colunas_preco].round(2)
analise_fabricante.to_csv(RESULTS_DIR / "analise_preco_fabricante.csv", index=False)
analise_fabricante.head(10)


## 7. Normalizacao por peso


In [ ]:
base_normalizada = base_marcas.copy()
quantidades = base_normalizada["titulo"].apply(extrair_quantidade)
base_normalizada[["peso_g", "unidade_peso_extraida", "unidades"]] = pd.DataFrame(
    quantidades.tolist(),
    index=base_normalizada.index,
)
base_normalizada["categoria_quantidade"] = np.select(
    [base_normalizada["peso_g"].notna(), base_normalizada["unidades"].notna()],
    ["peso", "unidades"],
    default="sem_quantidade",
)
base_normalizada["faixa_peso"] = base_normalizada["peso_g"].apply(faixa_peso)
base_normalizada["preco_por_100g"] = np.where(
    base_normalizada["peso_g"].notna() & (base_normalizada["peso_g"] > 0),
    base_normalizada["preco"] / base_normalizada["peso_g"] * 100,
    np.nan,
)
base_normalizada["preco_por_500g"] = np.where(
    base_normalizada["peso_g"].notna() & (base_normalizada["peso_g"] > 0),
    base_normalizada["preco"] / base_normalizada["peso_g"] * 500,
    np.nan,
)
base_normalizada["preco_por_unidade"] = np.where(
    base_normalizada["unidades"].notna() & (base_normalizada["unidades"] > 0),
    base_normalizada["preco"] / base_normalizada["unidades"],
    np.nan,
)

colunas_arredondar = ["preco", "peso_g", "unidades", "preco_por_100g", "preco_por_500g", "preco_por_unidade"]
base_normalizada[colunas_arredondar] = base_normalizada[colunas_arredondar].round(2)
base_normalizada.to_csv(BASE_DIR / "base_cafe_normalizada_peso.csv", index=False)
base_normalizada.head()


In [ ]:
analise_preco_normalizado = (
    base_normalizada
    .groupby("loja")
    .agg(
        produtos=("sku", "count"),
        produtos_com_peso=("peso_g", lambda s: int(s.notna().sum())),
        cobertura_peso_pct=("peso_g", lambda s: round(s.notna().mean() * 100, 2)),
        preco_mediano=("preco", "median"),
        preco_500g_mediano=("preco_por_500g", "median"),
        preco_100g_mediano=("preco_por_100g", "median"),
        preco_unidade_mediano=("preco_por_unidade", "median"),
    )
    .reset_index()
)
colunas = ["preco_mediano", "preco_500g_mediano", "preco_100g_mediano", "preco_unidade_mediano"]
analise_preco_normalizado[colunas] = analise_preco_normalizado[colunas].round(2)
analise_preco_normalizado.to_csv(RESULTS_DIR / "analise_preco_normalizado_peso_loja.csv", index=False)
analise_preco_normalizado


In [ ]:
analise_faixa_peso = (
    base_normalizada
    .groupby(["loja", "faixa_peso"])
    .agg(
        produtos=("sku", "count"),
        preco_mediano=("preco", "median"),
        preco_500g_mediano=("preco_por_500g", "median"),
    )
    .reset_index()
)
analise_faixa_peso[["preco_mediano", "preco_500g_mediano"]] = analise_faixa_peso[
    ["preco_mediano", "preco_500g_mediano"]
].round(2)
analise_faixa_peso.to_csv(RESULTS_DIR / "analise_faixa_peso_loja.csv", index=False)
analise_faixa_peso


## 8. Evolucao temporal

A evolucao temporal usa apenas a base do Pao de Acucar, que possui tres dias de historico.


In [ ]:
evolucao_preco = (
    base_cafe
    .groupby("Data")[["Média Preço Normal", "Média Preço Oferta"]]
    .mean()
    .round(2)
    .reset_index()
)
evolucao_preco["Desvio_Preco"] = (
    base_cafe
    .groupby("Data")["Média Preço Normal"]
    .std()
    .round(2)
    .values
)
evolucao_preco.to_csv(RESULTS_DIR / "evolucao_preco_cafe.csv", index=False)
evolucao_preco


## 9. Geracao dos graficos

Esta celula executa o script que cria os PNGs em `results/graficos`.


In [ ]:
runpy.run_path("scripts/gerar_graficos.py", run_name="__main__")


## 10. Conferencia final


In [ ]:
for arquivo in sorted(RESULTS_DIR.glob("*.csv")):
    df = pd.read_csv(arquivo)
    print(f"{arquivo.name}: {df.shape}")

print()
print("Graficos:")
for arquivo in sorted((RESULTS_DIR / "graficos").glob("*.png")):
    print("-", arquivo.name)
